In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from keras import regularizers, layers
import matplotlib.pyplot as plt
import time
import numpy as np
import os
from IPython import display
import shutil

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
(train_images, train_labels), (_, _) = tf.keras.datasets.cifar10.load_data()

In [ ]:
def preprocess_dataset(images, labels):
    # Normalize images to [-1, 1]
    images = tf.cast(images, tf.float32)
    images = (images - 127.5) / 127.5
    labels = tf.reshape(labels, shape=(1,))
    return images, labels

# Apply preprocessing and batch
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.map(lambda x, y: preprocess_dataset(x, y)).batch(64)
train_dataset = train_dataset.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE).shuffle(buffer_size=train_dataset.cardinality(), reshuffle_each_iteration=True)


In [ ]:
def make_discriminator_model(in_shape=(32, 32, 3), n_classes=10):
    # Image input
    in_image = layers.Input(shape=in_shape)
    in_image = layers.GaussianNoise(0.1)(in_image)

    # One-hot encoded label input
    in_label = layers.Input(shape=(1,))  # One-hot encoded labels (None, 10)
    label = layers.Embedding(n_classes, 50)(in_label)

    # Project one-hot labels into a dense representation and reshape
    y = layers.Dense(in_shape[0] * in_shape[1], use_bias=False)(label)
    y = layers.Reshape((in_shape[0], in_shape[1], 1))(y)

    # Merge image and label feature maps
    merge = layers.Concatenate()([in_image, y])

    # Convolution
    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(merge)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02, axis=-1)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02, axis=-1)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(32, (5, 5), strides=(2, 2), padding='same', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02, axis=-1)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(1, activation='sigmoid')(x)

    # Define model
    model = keras.Model([in_image, in_label], outputs)
    return model

In [ ]:
def make_generator_model(noise_dim = 100, num_classes = 10):

    # Input noise
    noise_input = layers.Input(shape=(noise_dim,))
    
    # Label input
    label_input = layers.Input(shape=(1,))
    label_embedding = layers.Embedding(num_classes, 50)(label_input)
    label = layers.Dense(4 * 4)(label_embedding)
    label = layers.Reshape((4,4, 1))(label)
    
    # Concatenate noise and embedded labels
    gen = layers.Dense(4*4*256)(noise_input)
    gen = layers.LeakyReLU(alpha=0.2)(gen)
    gen = layers.Reshape((4, 4, 256))(gen)

    # Generator network
    merge = layers.Concatenate()([gen, label])

    x = layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False, kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(merge)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02, axis=-1)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False, kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02, axis=-1)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False, kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, center=1.0, scale=0.02, axis=-1)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    image_output = layers.Conv2DTranspose(3, (5, 5), strides=(1, 1), padding='same', activation='tanh', kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02))(x)
    
    # Define the model
    generator = keras.Model([noise_input, label_input], image_output)
    return generator

In [ ]:
generator = make_generator_model()
discriminator = make_discriminator_model()

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=False)

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

In [ ]:
def generator_loss(fake_output, true_labels):
    adv_loss = tf.keras.losses.BinaryCrossentropy(from_logits=False)(tf.ones_like(fake_output), fake_output)
    return adv_loss

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5, beta_2=0.999)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0003, beta_1=0.5, beta_2=0.999)

In [ ]:
EPOCHS = 101
noise_dim = 100
num_examples_to_generate = 16

seed = tf.random.normal([num_examples_to_generate, noise_dim])

In [ ]:
@tf.function
def train_step(images, labels):
    batch_size = tf.shape(labels)[0]  # Get the batch size from the labels
    noise = tf.random.normal([batch_size, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator([noise, labels], training=True)

        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        gen_loss = generator_loss(fake_output, labels)
        disc_loss = discriminator_loss(real_output, fake_output)

    # Apply gradients to the discriminator
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    # Apply gradients to the generator (first update)
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    
    batch_size = tf.shape(labels)[0]  # Get the batch size from the labels
    noise = tf.random.normal([batch_size, 100])
    
    # Second gradient computation for generator
    with tf.GradientTape() as second_gen_tape:
        # Regenerate fake images and labels
        generated_images = generator([noise, labels], training=True)
        
        fake_output = discriminator([generated_images, labels], training=True)  # Reuse discriminator for fake_output
        
        # Compute loss again
        second_gen_loss = generator_loss(fake_output, labels)

    # Apply second set of gradients to the generator
    second_gradients_of_generator = second_gen_tape.gradient(second_gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(second_gradients_of_generator, generator.trainable_variables))

In [ ]:
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        for image_batch, label_batch in dataset:
            train_step(image_batch, label_batch)

        # Example of generating test labels
        batch_size = 16
        test_labels = generate_random_labels(batch_size)


        # Produce images for the GIF as you go
        display.clear_output(wait=True)
        generate_and_save_images(generator,
                                 epoch + 1,
                                 seed,test_labels)

        # Save the model every 15 epochs
        if epoch == 100:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print('Time for epoch {} is {} sec'.format(epoch + 1, time.time() - start))

    # Generate after the final epoch
    display.clear_output(wait=True)
    generate_and_save_images(generator,
                             epochs,
                             seed,test_labels)

In [ ]:
def generate_random_labels(batch_size, num_classes=10):
    # Generate random integers between 1 and num_classes (inclusive)
    labels = tf.random.uniform(
        shape=(batch_size, 1),
        minval=0,
        maxval=num_classes,  # maxval is exclusive
        dtype=tf.int32
    )
    return labels

In [ ]:
def generate_and_save_images(model, epoch, test_input, test_labels):

    # Pass both inputs to the generator
    predictions = model([test_input, test_labels], training=False)

    # Plot generated images
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5)
        plt.axis('off')

    plt.show()


In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
train(train_dataset, EPOCHS)

In [ ]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [ ]:
class_labels = {
    0: 'airplane', 1: 'automobile', 2: 'bird', 3: 'cat', 4: 'deer',
    5: 'dog', 6: 'frog', 7: 'horse', 8: 'ship', 9: 'truck'
}

def save_images_with_labels(generator, num_images, folder, batch_size=1000):
    os.makedirs(folder, exist_ok=True)

    for batch_start in range(0, num_images, batch_size):
        current_batch_size = min(batch_size, num_images - batch_start)

        # Generate noise and labels
        noise = np.random.normal(0, 1, (current_batch_size, 100))
        noise = tf.convert_to_tensor(noise, dtype=tf.float32)  # Convert to tensor
        
        test_labels = generate_random_labels(current_batch_size)
        
        # Generate images and labels
        generated_images = generator([noise, test_labels])
        
        # Convert predicted labels to class indices
        predicted_labels = test_labels.numpy()

        for i, (image, label) in enumerate(zip(generated_images.numpy(), predicted_labels)):

            label = int(label)
            label_name = class_labels[label]

            # Normalize to [0, 1]
            image_rgb = np.clip(image, 0, 1)

            # Save image
            image_index = batch_start + i
            save_path = f'{folder}/generated_image_{image_index}_label_{label_name}.png'
            plt.imsave(save_path, image_rgb.reshape(32, 32, 3))

            # Move to "good images" folder
            shutil.move(save_path, os.path.join(folder, os.path.basename(save_path)))


In [ ]:
save_dir = "generated_images"
save_images_with_labels(generator, num_images=100000, folder=save_dir)

In [ ]:
# Paths for source and target directories
image_dir = '/kaggle/working/generated_images'
parent_dir = 'images_png'

# Create the parent directory if it doesn't exist
os.makedirs(parent_dir, exist_ok=True)

# Loop through all image files in the source directory
for image_file in os.listdir(image_dir):
    if image_file.endswith('.png'):
        try:
            # Extract label from filename (assumes the label is after "label_" in the filename)
            label = image_file.split('_label_')[-1].split('_')[0]

            # Create the directory for the label if it doesn't exist
            label_dir = os.path.join(parent_dir, label)
            os.makedirs(label_dir, exist_ok=True)

            # Move the file to the corresponding label directory
            source_path = os.path.join(image_dir, image_file)
            target_path = os.path.join(label_dir, image_file)

            # Check for duplicate files and rename if necessary
            if os.path.exists(target_path):
                base, ext = os.path.splitext(image_file)
                target_path = os.path.join(label_dir, f"{base}_duplicate{ext}")

            shutil.move(source_path, target_path)

        except Exception as e:
            # Log any errors and continue
            print(f"Error processing file {image_file}: {e}")
